<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 4.1 — Regression as an Optimization Problem

In M4.1 you saw that fitting a regression line is an optimization problem: choose the coefficients
that make the total error as small as possible. Tonight we use the same pharmacy data and look at the
**three things you control** when you write the model yourself:

| Knob | Where it lives in the model | Tonight's options |
|---|---|---|
| 1. The **form** of the model | the constraints (one per row of data) | linear: `a + b*hours`, or power: `a * hours**b` |
| 2. The **error** you minimize | the objective | squared error, or absolute error |
| 3. **Business rules** | extra constraints | finance wants a budget-safe line: never predict more than a store actually made |

The third knob is the one ordinary regression software does not give you. With an optimization model,
a business rule is one more line of code.

**How this notebook works.** Everything is already written. You change the **SCENARIO** cell, then
click on it and choose **Runtime → Run cell and below** (Ctrl+F10). Every scenario you run is saved
on a scoreboard, and the last cell draws all of them on one picture.

## Import modules (download Pyomo and the ipopt solver)

Run this first, every time. It installs Pyomo and **ipopt**, the nonlinear solver we use in Module 4.
It takes about a minute.

In [ ]:
# install Pyomo and the ipopt solver in Colab (idaes-pse brings both)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

# this makes plots using pylab (matplotlib)
%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd

# import ALL NAMESPACES from pyomo
from pyomo.environ import *

# where the setup cell put the solver
ipopt_path = '/content/bin/ipopt'

# Part 1 · The data

Ten pharmacies: how many hours a week each one is open, and its weekly revenue. Same data as M4.1.

In [ ]:
# read the data
pharmacy_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/7_Nonlinear/data/pharmacy.csv')
pharmacy_df

In [ ]:
# soft-code the data into dictionaries, keyed by store number
stores = list(pharmacy_df['store'])

hours = {}
revenue = {}
for i in range(len(pharmacy_df)):
    store = pharmacy_df.loc[i, 'store']
    hours[store] = float(pharmacy_df.loc[i, 'hours'])
    revenue[store] = float(pharmacy_df.loc[i, 'revenue'])

print("hours:", hours)
print("revenue:", revenue)

In [ ]:
# look before you model
scatter(pharmacy_df['hours'], pharmacy_df['revenue'])
xlabel('Hours open per week')
ylabel('Weekly revenue ($)')
title('Ten pharmacies')
show()

**The answer to check against.** `scipy` fits the ordinary least-squares line in one line of code.
Our base scenario must give exactly the same `a` and `b`. If it does, the optimization model is right.

In [ ]:
from scipy import stats

scipy_fit = stats.linregress(pharmacy_df['hours'], pharmacy_df['revenue'])
print("scipy:  a = %.2f   b = %.4f" % (scipy_fit.intercept, scipy_fit.slope))

**The scoreboard.** An empty dictionary. Each scenario you run adds one entry:
`scoreboard[scenario] = what the model found`. Run this cell **once**.

In [ ]:
# scenario name -> what the model found
scoreboard = {}

# Part 2 · The scenario

Leave it alone for the first run (the base scenario). Then try the others **one at a time**: remove
the `#` from that scenario's lines, click on the cell, and choose **Runtime → Run cell and below**.
Put the `#` back when you are done.

In [ ]:
# ---------- THE SCENARIO (change things here) ----------
scenario = "1 base: linear, squared error"
model_form = "linear"      # "linear" (a + b*hours)  or  "power" (a * hours**b)
error_type = "squared"     # "squared"  or  "absolute"
never_overpredict = False  # business rule: the prediction for every store must be at or below what it actually made

# Scenario 2: absolute error instead of squared error
# scenario = "2 absolute error"
# error_type = "absolute"

# Scenario 3: a power model, revenue = a * hours**b
# scenario = "3 power model"
# model_form = "power"

# Scenario 4: finance wants a budget-safe forecast that never promises more than a store made
# scenario = "4 rule: never over-predict"
# never_overpredict = True

# Scenario 5: your own idea (combine knobs! e.g. a power model that never over-predicts)


# Part 3 · The model

**3a · Decision variables.** The two coefficients, `a` and `b`, plus one **placeholder** variable
per store: `model.y[store]` is the model's predicted revenue for that store. That is the pattern from
M4.1: ten stores, ten predictions.

In [ ]:
# ---------- 3a. the decision variables ----------
model = ConcreteModel()

model.a = Var(domain=Reals)
model.b = Var(domain=Reals)
model.y = Var(stores, domain=Reals)     # one predicted revenue per store

# nonlinear solvers need a sensible starting point (the hill climber in the fog!)
if model_form == "linear":
    model.a = 1000
    model.b = 10
else:
    model.a = 1000
    model.b = 0.5

**3b · Knob 1, the form of the model: one constraint per store.** This is where the regression
equation lives: not in the objective, in the constraints. Each constraint says "the prediction for this
store must follow the model".

In [ ]:
# ---------- 3b. knob 1: the form of the model (one constraint per store) ----------
model.constraints = ConstraintList()

for store in stores:
    if model_form == "linear":
        model.constraints.add(model.y[store] == model.a + model.b * hours[store])
    else:
        model.constraints.add(model.y[store] == model.a * hours[store] ** model.b)

print(model_form, "model:", len(stores), "prediction constraints")

**3c · Knob 2, the error: the objective.** Squared error is the usual choice. Absolute error
cannot use `abs()` (the solver needs a smooth slope, and a V has a sharp point), so we use the trick
from M4.1: one more variable per store, `model.d[store]`, with **two floors**: `d >= miss` and `d >= -miss`.

Why does that give the absolute value? One of the two floors is always the positive size of the miss:

| the store's miss (prediction - actual) | floor 1: d >= miss | floor 2: d >= -miss | smallest d allowed |
|---|---|---|---|
| +300 (we over-predicted) | d >= 300 | d >= -300 | **300** |
| -200 (we under-predicted) | d >= -200 | d >= 200 | **200** |

We are **minimizing** the total of the `d`'s, so the solver pushes every `d` down until it hits the higher floor,
which is exactly the size of the miss. (It only works because we minimize: if we maximized, `d` would run off
to infinity.)

In [ ]:
# ---------- 3c. knob 2: the error (the objective) ----------
obj_expr = 0

if error_type == "squared":
    for store in stores:
        obj_expr += (model.y[store] - revenue[store]) ** 2
else:
    model.d = Var(stores, domain=NonNegativeReals)     # d[store] will equal |error| at the optimum
    for store in stores:
        model.constraints.add(model.d[store] >= model.y[store] - revenue[store])
        model.constraints.add(model.d[store] >= -(model.y[store] - revenue[store]))
        obj_expr += model.d[store]

model.error = Objective(expr=obj_expr, sense=minimize)
print("minimizing", error_type, "error")

**3d · Knob 3, a business rule: never over-predict.** Finance is building next year's budget and wants a
revenue line that never promises more than a store actually made: over-forecasting means hiring staff and
buying stock for sales that never come. That is one more constraint per store: `prediction <= actual`.
Plain regression has no way to say this. Here it is a loop, and it works for either model form.

In [ ]:
# ---------- 3d. knob 3: a business rule ----------
if never_overpredict:
    for store in stores:
        model.constraints.add(model.y[store] <= revenue[store])
    print("rule: never over-predict (", len(stores), "more constraints )")
else:
    print("no business rule in this scenario")

**3e · Solve it and read the results.** Whatever error you minimized, we also report the total
squared error, so every scenario can be compared on the same yardstick. And we predict a store open
120 hours a week, a store we have no data for.

In [ ]:
# ---------- 3e. solve and report ----------
results = SolverFactory('ipopt', executable=ipopt_path).solve(model)
print("SCENARIO:", scenario)
print("Termination:", results.solver.termination_condition)

a = model.a()
b = model.b()
if model_form == "linear":
    print("revenue = %.2f + %.4f * hours" % (a, b))
else:
    print("revenue = %.2f * hours ** %.4f" % (a, b))

# the same yardsticks for every scenario
sse = 0
sae = 0
for store in stores:
    sse += (model.y[store]() - revenue[store]) ** 2
    sae += abs(model.y[store]() - revenue[store])

if model_form == "linear":
    pred_120 = a + b * 120
else:
    pred_120 = a * 120 ** b

print("total squared error  = {:,.0f}".format(sse))
print("total absolute error = {:,.0f}".format(sae))
print("predicted revenue for a store open 120 hours = ${:,.0f}".format(pred_120))

# scoreboard[key] = value
scoreboard[scenario] = {'form': model_form, 'a': a, 'b': b, 'squared error': sse, 'absolute error': sae, 'revenue at 120 hours': pred_120}

# Part 4 · The scoreboard

Every scenario you ran, in one table and one picture. Run this after each scenario.

In [ ]:
# the scoreboard as a table
pd.DataFrame(scoreboard).T.round(2)

In [ ]:
# every scenario's curve on one picture
grid = np.linspace(0, 175, 200)

scatter(pharmacy_df['hours'], pharmacy_df['revenue'], color='black', zorder=3, label='the ten stores')
for name in scoreboard:
    fit = scoreboard[name]
    if fit['form'] == 'linear':
        curve = fit['a'] + fit['b'] * grid
    else:
        curve = fit['a'] * grid ** fit['b']
    plot(grid, curve, label=name)
xlabel('Hours open per week')
ylabel('Weekly revenue ($)')
title('Same data, different choices')
legend(fontsize=8)
show()

### Your turn

Run scenarios 1 to 4, then answer in a text cell:

1. Does the base scenario match `scipy` exactly? Why does that matter?
2. Absolute error vs. squared error: how did `a` and `b` change? Squared error punishes one big miss much more
   than several small ones. Which error measure lets a few stores pull the line around more?
3. The power model: is `b` above or below 1? What does that say about the value of staying open longer?
4. The never-over-predict rule makes the squared error worse. Why is that guaranteed, and why would finance still want it?
5. Scenario 5: combine knobs (for example, a power model that never over-predicts). What did you learn?

## Bottom line

A regression is an optimization model with three parts you choose: the **form** (constraints), the
**error** (objective), and any **rules** (more constraints). Software like `scipy` makes the first two
choices for you. When you write the model yourself, you can add the business knowledge the data
doesn't have.